In [1]:
from pyspark.sql import SparkSession


In [2]:
# Create Spark Session
spark = SparkSession.builder \
    .appName("Retail Demand Forecasting") \
    .master("local[*]") \
    .getOrCreate()



In [3]:

sc = spark.sparkContext


In [4]:

customers = spark.read.csv(
    "/content/olist_customers_dataset.csv",
    header=True,
    inferSchema=True
)

orders = spark.read.csv(
    "/content/olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)

order_items = spark.read.csv(
    "/content/olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)

products = spark.read.csv(
    "/content/olist_products_dataset.csv",
    header=True,
    inferSchema=True
)

sellers = spark.read.csv(
    "/content/olist_sellers_dataset.csv",
    header=True,
    inferSchema=True
)

payments = spark.read.csv(
    "/content/olist_order_payments_dataset.csv",
    header=True,
    inferSchema=True
)

reviews = spark.read.csv(
    "/content/olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=True
)

geolocation = spark.read.csv(
    "/content/olist_geolocation_dataset.csv",
    header=True,
    inferSchema=True
)

category_translation = spark.read.csv(
    "/content/product_category_name_translation.csv",
    header=True,
    inferSchema=True
)



In [5]:
print("Customers :", customers.count())
print("Orders :", orders.count())
print("Order Items :", order_items.count())
print("Products :", products.count())
print("Sellers :", sellers.count())
print("Payments :", payments.count())
print("Reviews :", reviews.count())
print("Geolocation :", geolocation.count())
print("Category Translation :", category_translation.count())

Customers : 99441
Orders : 99441
Order Items : 112650
Products : 32951
Sellers : 3095
Payments : 103886
Reviews : 104162
Geolocation : 1000163
Category Translation : 71


In [6]:
orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [7]:
customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [8]:
products.printSchema()


root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [9]:
order_items.printSchema()


root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



In [10]:
payments.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



In [11]:

reviews.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)



In [12]:
sellers.printSchema()

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



In [13]:
from pyspark.sql.functions import col, count, when

datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Products": products,
    "Sellers": sellers,
    "Payments": payments,
    "Reviews": reviews,
    "Geolocation": geolocation,
    "Category Translation": category_translation
}

for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"Missing Values in {name}")
    print(f"{'='*60}")

    df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]).show()


Missing Values in Customers
+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+


Missing Values in Orders
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|    

In [14]:
print(customers.columns)
print(products.columns)
print(order_items.columns)
print(orders.columns)

['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [15]:
customers.dtypes

orders.dtypes

products.dtypes

order_items.dtypes

[('order_id', 'string'),
 ('order_item_id', 'int'),
 ('product_id', 'string'),
 ('seller_id', 'string'),
 ('shipping_limit_date', 'timestamp'),
 ('price', 'double'),
 ('freight_value', 'double')]

In [16]:
# Cache Frequently Used Tables
orders.cache()
order_items.cache()
products.cache()
customers.cache()

DataFrame[customer_id: string, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string]

In [17]:
# @title Q2


In [18]:
orders_rdd = orders.rdd
order_items_rdd = order_items.rdd
products_rdd = products.rdd
customers_rdd = customers.rdd

In [19]:
orders_rdd.take(5)

[Row(order_id='e481f51cbdc54678b7cc49136f2d6af7', customer_id='9ef432eb6251297304e76186b10a928d', order_status='delivered', order_purchase_timestamp=datetime.datetime(2017, 10, 2, 10, 56, 33), order_approved_at=datetime.datetime(2017, 10, 2, 11, 7, 15), order_delivered_carrier_date=datetime.datetime(2017, 10, 4, 19, 55), order_delivered_customer_date=datetime.datetime(2017, 10, 10, 21, 25, 13), order_estimated_delivery_date=datetime.datetime(2017, 10, 18, 0, 0)),
 Row(order_id='53cdb2fc8bc7dce0b6741e2150273451', customer_id='b0830fb4747a6c6d20dea0b8c802d7ef', order_status='delivered', order_purchase_timestamp=datetime.datetime(2018, 7, 24, 20, 41, 37), order_approved_at=datetime.datetime(2018, 7, 26, 3, 24, 27), order_delivered_carrier_date=datetime.datetime(2018, 7, 26, 14, 31), order_delivered_customer_date=datetime.datetime(2018, 8, 7, 15, 27, 45), order_estimated_delivery_date=datetime.datetime(2018, 8, 13, 0, 0)),
 Row(order_id='47770eb9100c2d0c44946d9cf07ec65d', customer_id='41ce

In [20]:
# Transformation – Filter Delivered Orders
delivered_orders = orders_rdd.filter(
    lambda x: x.order_status == "delivered"
)


delivered_orders.count()

96478

In [21]:
# Transformation – Filter Expensive Products
expensive_products = order_items_rdd.filter(
    lambda x: x.price > 500
)

expensive_products.take(10)

[Row(order_id='000576fe39319847cbb9d288c5617fa6', order_item_id=1, product_id='557d850972a7d6f792fd18ae1400d9b6', seller_id='5996cddab893a4652a15592fb58ab8db', shipping_limit_date=datetime.datetime(2018, 7, 10, 12, 30, 45), price=810.0, freight_value=70.75),
 Row(order_id='0009c9a17f916a706d71784483a5d643', order_item_id=1, product_id='3f27ac8e699df3d300ec4a5d8c5cf0b2', seller_id='fcb5ace8bcc92f75707dc0f01a27d269', shipping_limit_date=datetime.datetime(2018, 5, 2, 9, 31, 53), price=639.0, freight_value=11.34),
 Row(order_id='0017afd5076e074a48f1f1a4c7bac9c5', order_item_id=1, product_id='fe59a1e006df3ac42bf0ceb876d70969', seller_id='25c5c91f63607446a97b143d2d535d31', shipping_limit_date=datetime.datetime(2017, 4, 24, 22, 25, 19), price=809.1, freight_value=44.29),
 Row(order_id='006127b8b9a1681a982313ed7129c3c0', order_item_id=1, product_id='f819f0c84a64f02d3a5606ca95edd272', seller_id='7d13fca15225358621be4086e1eb0964', shipping_limit_date=datetime.datetime(2018, 3, 15, 2, 30, 23), pr

In [22]:
# Transformation – Map
product_ids = order_items_rdd.map(
    lambda x: x.product_id
)
product_ids.take(10)

['4244733e06e7ecb4970a6e2683c13e61',
 'e5f2d52b802189ee658865ca93d83a8f',
 'c777355d18b72b67abbeef9df44fd0fd',
 '7634da152a4610f1595efa32f14722fc',
 'ac6c3623068f30de03045865e4e10089',
 'ef92defde845ab8450f9d70c526ef70f',
 '8d4f2bb7e93e6710a28f34fa83ee7d28',
 '557d850972a7d6f792fd18ae1400d9b6',
 '310ae3c140ff94b03219ad0adc3c778f',
 '4535b0e1091c278dfd193e5a1d63b39f']

In [23]:
# Transformation – Distinct
unique_products = product_ids.distinct()
unique_products.count()

32951

In [24]:
# Transformation – Sort
sorted_products = order_items_rdd.sortBy(
    lambda x: x.price,
    ascending=False
)
sorted_products.take(10)


[Row(order_id='0812eb902a67711a1cb742b3cdaa65ae', order_item_id=1, product_id='489ae2aa008f021502940f251d4cce7f', seller_id='e3b4998c7a498169dc7bce44e6bb6277', shipping_limit_date=datetime.datetime(2017, 2, 16, 20, 37, 36), price=6735.0, freight_value=194.31),
 Row(order_id='fefacc66af859508bf1a7934eab1e97f', order_item_id=1, product_id='69c590f7ffc7bf8db97190b6cb6ed62e', seller_id='80ceebb4ee9b31afb6c6a916a574a1e2', shipping_limit_date=datetime.datetime(2018, 8, 2, 4, 5, 13), price=6729.0, freight_value=193.21),
 Row(order_id='f5136e38d1a14a4dbd87dff67da82701', order_item_id=1, product_id='1bdf5e6731585cf01aa8169c7028d6ad', seller_id='ee27a8f15b1dded4d213a468ba4eb391', shipping_limit_date=datetime.datetime(2017, 6, 15, 2, 45, 17), price=6499.0, freight_value=227.66),
 Row(order_id='a96610ab360d42a2e5335a3998b4718a', order_item_id=1, product_id='a6492cc69376c469ab6f61d8f44de961', seller_id='59417c56835dd8e2e72f91f809cd4092', shipping_limit_date=datetime.datetime(2017, 4, 18, 13, 25, 18

In [25]:
# Transformation – Union
customers_small = customers_rdd.take(5)
sellers_small = sellers.rdd.take(5)
# Convert back into RDDs.
customers_small_rdd = sc.parallelize(customers_small)
sellers_small_rdd = sc.parallelize(sellers_small)

combined_rdd = customers_small_rdd.union(sellers_small_rdd)
combined_rdd.collect() #only on small dataset brings all data to driver

[Row(customer_id='06b8999e2fba1a1fbc88172c00ba8bc7', customer_unique_id='861eff4711a542e4b93843c6dd7febb0', customer_zip_code_prefix=14409, customer_city='franca', customer_state='SP'),
 Row(customer_id='18955e83d337fd6b2def6b18a428ac77', customer_unique_id='290c77bc529b7ac935b93aa66c333dc3', customer_zip_code_prefix=9790, customer_city='sao bernardo do campo', customer_state='SP'),
 Row(customer_id='4e7b3e00288586ebd08712fdd0374a03', customer_unique_id='060e732b5b29e8181a18229c7b0b2b5e', customer_zip_code_prefix=1151, customer_city='sao paulo', customer_state='SP'),
 Row(customer_id='b2b6027bc5c5109e529d4dc6358b12c3', customer_unique_id='259dac757896d24d7702b9acbbff3f3c', customer_zip_code_prefix=8775, customer_city='mogi das cruzes', customer_state='SP'),
 Row(customer_id='4f2d8ab171c80ec8364f7c12e35b23ad', customer_unique_id='345ecd01c38d18a9036ed96c73b8d066', customer_zip_code_prefix=13056, customer_city='campinas', customer_state='SP'),
 Row(seller_id='3442f8959a84dea7ee197c632cb2

In [26]:
# @title Q3


In [27]:
# Key-Value Pair RDD
# Product ID as Key, Price as Value
product_price_rdd = order_items_rdd.map(lambda x: (x.product_id, x.price))

product_price_rdd.take(5)

[('4244733e06e7ecb4970a6e2683c13e61', 58.9),
 ('e5f2d52b802189ee658865ca93d83a8f', 239.9),
 ('c777355d18b72b67abbeef9df44fd0fd', 199.0),
 ('7634da152a4610f1595efa32f14722fc', 12.99),
 ('ac6c3623068f30de03045865e4e10089', 199.9)]

In [28]:
# Shuffle Operation using reduceByKey()
product_sales = product_price_rdd.reduceByKey(lambda x, y: x + y)

product_sales.take(10)

[('4244733e06e7ecb4970a6e2683c13e61', 533.1),
 ('c777355d18b72b67abbeef9df44fd0fd', 597.0),
 ('7634da152a4610f1595efa32f14722fc', 25.98),
 ('557d850972a7d6f792fd18ae1400d9b6', 810.0),
 ('310ae3c140ff94b03219ad0adc3c778f', 291.9),
 ('4535b0e1091c278dfd193e5a1d63b39f', 215.96),
 ('d63c1011f49d98b976c352955b1c4bea', 3259.45),
 ('f177554ea93259a5b282f24e33f65ab6', 135.0),
 ('368c6c730842d78016ad823897a372db', 21056.799999999937),
 ('3f27ac8e699df3d300ec4a5d8c5cf0b2', 639.0)]

In [29]:
# Demonstrate RDD Persistence
# Store the RDD in memory for faster reuse

product_sales.cache()


PythonRDD[288] at RDD at PythonRDD.scala:56

In [30]:
# Step 4: Verify Storage Level
print(product_sales.getStorageLevel())

Memory Serialized 1x Replicated


In [31]:
# @title Q4
#  Selection Operation
# conversion happpened in q1


products.select(
    "product_id",
    "product_category_name"
).show(5, truncate=False)

+--------------------------------+---------------------+
|product_id                      |product_category_name|
+--------------------------------+---------------------+
|1e9e8ef04dbcff4541ed26657ea517e5|perfumaria           |
|3aa071139cb16b67ca9e5dea641aaa2f|artes                |
|96bd76ec8810374ed1b65e291975717f|esporte_lazer        |
|cef67bcfe19066a932b7673e239eb23d|bebes                |
|9dc1a7de274444849c219cff195d0b71|utilidades_domesticas|
+--------------------------------+---------------------+
only showing top 5 rows


In [32]:
#  Join Operation

# Join Products with Order Items.

product_sales = order_items.join(
    products,
    on="product_id",
    how="inner"
)

product_sales.select(
    "product_id",
    "product_category_name",
    "price"
).show(5)

+--------------------+---------------------+-----+
|          product_id|product_category_name|price|
+--------------------+---------------------+-----+
|4244733e06e7ecb49...|           cool_stuff| 58.9|
|e5f2d52b802189ee6...|             pet_shop|239.9|
|c777355d18b72b67a...|     moveis_decoracao|199.0|
|7634da152a4610f15...|           perfumaria|12.99|
|ac6c3623068f30de0...|   ferramentas_jardim|199.9|
+--------------------+---------------------+-----+
only showing top 5 rows


In [33]:
# GroupBy and Aggregation

# Calculate total sales for each product category.

from pyspark.sql.functions import sum

category_sales = product_sales.groupBy(
    "product_category_name"
).agg(
    sum("price").alias("Total_Sales")
)

category_sales.show(10)

+---------------------+------------------+
|product_category_name|       Total_Sales|
+---------------------+------------------+
|                  pcs|222963.12999999995|
|                bebes|411764.88999999495|
|                artes|24202.639999999978|
|            cine_foto| 6933.460000000001|
|     moveis_decoracao| 729762.4900000065|
|             pc_gamer|           1545.95|
| construcao_ferram...|         144677.59|
| tablets_impressao...|           7528.41|
|    artigos_de_festas| 4485.180000000001|
| fashion_roupa_mas...|10797.819999999989|
+---------------------+------------------+
only showing top 10 rows


In [34]:
# @title Q5


In [35]:
# Create Temporary Views
orders.createOrReplaceTempView("orders")

order_items.createOrReplaceTempView("order_items")

products.createOrReplaceTempView("products")

sellers.createOrReplaceTempView("sellers")

customers.createOrReplaceTempView("customers")

category_translation.createOrReplaceTempView("category_translation")


'''Spark SQL cannot directly query DataFrames using SQL syntax.

So we register each DataFrame as a temporary SQL table.

After this, Spark treats them like database tables.'''

'Spark SQL cannot directly query DataFrames using SQL syntax.\n\nSo we register each DataFrame as a temporary SQL table.\n\nAfter this, Spark treats them like database tables.'

In [36]:
# Identify Top Selling Products
top_products = spark.sql("""
SELECT
    oi.product_id,
    SUM(oi.price) AS Total_Sales
FROM order_items oi
GROUP BY oi.product_id
ORDER BY Total_Sales DESC
LIMIT 10
""")

top_products.show()

+--------------------+------------------+
|          product_id|       Total_Sales|
+--------------------+------------------+
|bb50f2e236e5eea01...|           63885.0|
|6cdd53843498f9289...| 54730.20000000005|
|d6160fb7873f18409...|48899.340000000004|
|d1c427060a0f73f6b...| 47214.51000000006|
|99a4788cb24856965...|43025.560000000085|
|3dd2a17168ec895c7...| 41082.60000000005|
|25c38557cf793876c...| 38907.32000000001|
|5f504b3a1c75b73d6...|37733.899999999994|
|53b36df67ebb7c415...| 37683.42000000001|
|aca2eb7d00ea1a7b8...| 37608.90000000007|
+--------------------+------------------+



In [37]:
# Category-wise Sales
category_sales = spark.sql("""
SELECT
    ct.product_category_name_english AS Category,
    ROUND(SUM(oi.price),2) AS Total_Sales
FROM order_items oi
JOIN products p
ON oi.product_id = p.product_id
JOIN category_translation ct
ON p.product_category_name = ct.product_category_name
GROUP BY ct.product_category_name_english
ORDER BY Total_Sales DESC
""")

category_sales.show()

+--------------------+-----------+
|            Category|Total_Sales|
+--------------------+-----------+
|       health_beauty| 1258681.34|
|       watches_gifts| 1205005.68|
|      bed_bath_table| 1036988.68|
|      sports_leisure|  988048.97|
|computers_accesso...|  911954.32|
|     furniture_decor|  729762.49|
|          cool_stuff|  635290.85|
|          housewares|  632248.66|
|                auto|  592720.11|
|        garden_tools|  485256.46|
|                toys|   483946.6|
|                baby|  411764.89|
|           perfumery|  399124.87|
|           telephony|  323667.53|
|    office_furniture|   273960.7|
|          stationery|  230943.23|
|           computers|  222963.13|
|            pet_shop|  214315.41|
| musical_instruments|  191498.88|
|    small_appliances|  190648.58|
+--------------------+-----------+
only showing top 20 rows


In [38]:
# Supplier Performance
supplier_performance = spark.sql("""
SELECT
    seller_id,
    COUNT(order_id) AS Orders_Supplied,
    ROUND(SUM(price),2) AS Revenue
FROM order_items
GROUP BY seller_id
ORDER BY Revenue DESC
LIMIT 10
""")

supplier_performance.show()

+--------------------+---------------+---------+
|           seller_id|Orders_Supplied|  Revenue|
+--------------------+---------------+---------+
|4869f7a5dfa277a7d...|           1156|229472.63|
|53243585a1d6dc264...|            410|222776.05|
|4a3ca9315b744ce9f...|           1987|200472.92|
|fa1c13f2614d7b5c4...|            586|194042.03|
|7c67e1448b00f6e96...|           1364|187923.89|
|7e93a43ef30c4f03f...|            340|176431.87|
|da8622b14eb17ae28...|           1551|160236.57|
|7a67c85e85bb2ce85...|           1171|141745.53|
|1025f0e2d44d7041d...|           1428|138968.55|
|955fee9216a65b617...|           1499| 135171.7|
+--------------------+---------------+---------+



In [39]:
# Seasonal Demand Trends
seasonal_trends = spark.sql("""
SELECT
    MONTH(order_purchase_timestamp) AS Month,
    ROUND(SUM(price),2) AS Monthly_Sales
FROM orders o
JOIN order_items oi
ON o.order_id = oi.order_id
GROUP BY MONTH(order_purchase_timestamp)
ORDER BY Month
""")

seasonal_trends.show()

+-----+-------------+
|Month|Monthly_Sales|
+-----+-------------+
|    1|   1070343.23|
|    2|   1091481.73|
|    3|   1357557.74|
|    4|   1356574.98|
|    5|   1502588.82|
|    6|   1298162.91|
|    7|    1393538.7|
|    8|   1428658.01|
|    9|    624814.05|
|   10|    713727.09|
|   11|   1010271.37|
|   12|    743925.07|
+-----+-------------+



In [40]:
# Monthly Inventory Report
inventory_report = spark.sql("""
SELECT
    MONTH(o.order_purchase_timestamp) AS Month,
    COUNT(oi.product_id) AS Products_Sold
FROM orders o
JOIN order_items oi
ON o.order_id = oi.order_id
GROUP BY MONTH(o.order_purchase_timestamp)
ORDER BY Month
""")

inventory_report.show()

+-----+-------------+
|Month|Products_Sold|
+-----+-------------+
|    1|         9163|
|    2|         9623|
|    3|        11217|
|    4|        10659|
|    5|        12061|
|    6|        10661|
|    7|        11611|
|    8|        12158|
|    9|         4838|
|   10|         5685|
|   11|         8665|
|   12|         6309|
+-----+-------------+



In [41]:
# @title Q6


In [42]:
# Transform Phase
etl_data = order_items.join(
    products,
    on="product_id",
    how="inner"
).join(
    sellers,
    on="seller_id",
    how="inner"
)

# Select Required Columns
final_etl = etl_data.select(
    "product_id",
    "seller_id",
    "product_category_name",
    "price",
    "seller_city",
    "seller_state"
)

# Load Phase
final_etl.write.mode("overwrite").csv(
    "output/retail_etl_output",
    header=True
)

# Display Output
final_etl.show(5)

+--------------------+--------------------+---------------------+-----+-------------+------------+
|          product_id|           seller_id|product_category_name|price|  seller_city|seller_state|
+--------------------+--------------------+---------------------+-----+-------------+------------+
|4244733e06e7ecb49...|48436dade18ac8b2b...|           cool_stuff| 58.9|volta redonda|          SP|
|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|             pet_shop|239.9|    sao paulo|          SP|
|c777355d18b72b67a...|5b51032eddd242adc...|     moveis_decoracao|199.0|borda da mata|          MG|
|7634da152a4610f15...|9d7a1d34a50524090...|           perfumaria|12.99|       franca|          SP|
|ac6c3623068f30de0...|df560393f3a51e745...|   ferramentas_jardim|199.9|       loanda|          PR|
+--------------------+--------------------+---------------------+-----+-------------+------------+
only showing top 5 rows


In [43]:
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator



In [44]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler

# Data Preparation: Join relevant DataFrames and create feature vector
# Use order_items for price (label) and freight_value, and products for product-specific numerical features.
data = order_items.join(products, on="product_id", how="inner")

# Define feature columns and label column
feature_columns = [
    "freight_value",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]
label_column = "price"

# Drop rows with nulls in the selected feature or label columns to avoid issues
data = data.na.drop(subset=feature_columns + [label_column])

# Create a VectorAssembler to combine feature columns into a single "features" vector
assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

# Transform the data to include the features vector
assembled_data = assembler.transform(data)

# Split the data into training and test sets
train_data, test_data = assembled_data.randomSplit([0.8, 0.2], seed=42)

# Create Random Forest Model
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="price",
    numTrees=100,
    maxDepth=8,
    seed=42
)

# Train Model
rf_model = rf.fit(train_data)

# Predict
rf_predictions = rf_model.transform(test_data)

# Show Predictions
rf_predictions.select(
    "price",
    "prediction"
).show(10)

# Evaluate
evaluator = RegressionEvaluator(
    labelCol="price",
    predictionCol="prediction",
    metricName="rmse"
)

rf_rmse = evaluator.evaluate(rf_predictions)

print("Random Forest RMSE:", rf_rmse)

+-----+------------------+
|price|        prediction|
+-----+------------------+
| 58.9|143.49147054759837|
| 78.9|122.44865587956866|
|34.99| 73.94074740512298|
|34.99|120.18667162612662|
|32.98| 81.22805670718252|
|199.7|154.39465943213565|
|54.99| 65.63508164727983|
|79.99| 62.58099091066901|
| 16.0| 43.97667541702081|
|153.0|111.33402370434176|
+-----+------------------+
only showing top 10 rows
Random Forest RMSE: 147.06916474467425
